# SHAFE Tutorial
### SHACL Advanced Features using SPARQL for Entity Schema Bridges - Tutorial

This tutorial is intended as a first introduction into building semantic bridges, A-Box based n-to-m mappings (mappings from multiple instances to multiple instances in this case with the chance to introduce new relations between the instances), from one ontological design pattern into another. 

#### The first questions is why do we want to have such a tool and there are multiple reasons for this:
1. Ontologies often have slightly different design patterns such that their data description models dont conform exactly. Just matching those patterns (espacially if they used the same classes) often leads to semantic errors and logical inconsistencies, which lead to them not beeing inferrable with a reasoner or even worse not beeing capable of being quieried due to loops and other challenges. Mapping the classes to each other works sometimes and partially but can either lead to the same problem with inconsistencies or the challenge that some entites can only partially be expressed in a combined ontology. As Crosswalks are only the list of realized mappings, this also holds true for the larger Crosswalks. Thus we want a tool which can translated/bridge between two ontology design pattern, but only for those instances of the origin ontology, which do not cause logic inconsistencies in the second ontology. SHAFE bridges can do so by validating whether custom criterias are fullfilled.
2. Currently there is tool which can map from one model into another while also adapting some of the origin design pattern, such that it conforms to the target ontology design pattern. Crosswalks and mappings can align classes, hwoever this does not change or modify the relation and therefore the "semantic hierarchy" of instances (meaning that some hierarchy is even required in the instances, where this is much more defined by domains and ranges, rather than true topological hierachy as in the class level). Crosswalks are also just multiple 1-to-1 mappings, which can by chance lead to a n-to-m mapping. However they only map classes to each other, and dont differentiate between certain instances or have refined mappings between relations. SHAFE Bridges allow to have more complex n-to-m mappings where for example a design pattern A1-B1-C1-D1 with the respective relations A1toB1-B1toC1-C1toD1 could be changed into A2-C2-B2-D2 with its respective relations A2toC2-C2toB2-B2toD2. The letter describes the actual instance, while the changing number at the respective ends is intended to express a changing terminology.



## First Steps
First we need to create the relevant tables for creating a SHAFE-bridge, which is a table of all relevant Prefixes, a list of all relevant curies and their locally used labels (for better readability), a list representing the shape, which shall be validated, and finally the bridge which reinterprets a set of entities from one ontology model into another ontology model.

Then we will create the visualizaiton of the shape, such that errors can be spotted and corrected early, and then we generate the SHACL-SPARQL shape.

This example shape is later executed against a test data set with pyshacl, one of the few packages which support SHACL-AdvancedFeatures at the time of writing (the other is the Topbraid/Topquadrant SHACL API). As pyshacl does not natively returns a modified shacl graph, a wrapper function is written, such that we can only examine the differences between the original graph and the extended graph. Feel also free to analyse the extended graph.


But before this we need to install all relevant packages, which is done in the next block:

In [ ]:
# run this only once - to install the required packages
# please consider using a virtual environment - https://docs.python.org/3/library/venv.html

! pip install pandas
! pip install copy
! pip install networkx
! pip install numpy
! pip install re
! pip install pyshacl
! pip install rdflib

In [3]:
# now importing the required packages, run this cell every time you start the notebook

import sys
import pandas as pd
from copy import deepcopy
import networkx as nx
import re
from pyshacl import Validator
from rdflib import Graph
from rdflib.compare import to_isomorphic, graph_diff
from IPython.display import display, Markdown

### Prefixes
Now we define all the required Prefixes and namespaces, such that they can be integrated into the SHACL Shape and the integrated SPARQL Query. Saidly this can not be easily automated.

In [4]:
Prefix_df = pd.DataFrame({'Prefix': ['ex', 'rdf', 'rdfs', 'xsd'], 'namespace': ['http://example.org/ontology#', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#', 'http://www.w3.org/2000/01/rdf-schema#', 'http://www.w3.org/2001/XMLSchema#']})

In [ ]:
Prefix_df

### Classes and Relations used in the SHAFE Bridge
next we have to define all the classes and relations we want to use. As the classes can be quite hard to interprete in their curie format, we also use labels here, to atleast have a human readable documentation of the SHAFE Bridge.

In [ ]:
Class_df = pd.DataFrame({'label': ['Process', 'ProcessStep', 'Input', 'InputSettings', 'ChemicalInvestigation', 'Setup', 'realizedOccurent', 'Specimen', 'Experiment', 'ExperimentSetup', 'Sample', 'Parameters'], 
                          'curie': ['ex:Process', 'ex:ProcessStep', 'ex:Input', 'ex:InputSettings', 'ex:ChemicalInvestigation', 'ex:Setup', 'ex:realizedOccurent', 'ex:Specimen', 'ex:Experiment', 'ex:ExperimentSetup', 'ex:Sample', 'ex:Parameters']
                         })
Relation_df = pd.DataFrame({'label': ['isSome', 'hasPart', 'isSome', 'describes', 'isInput', 'hasModifier', 'is_a', 'hasExperimentSetup', 'hasSample', 'performedWith', 'hasConcentrations'], 
                          'curie': ['ex:isSome', 'ex:hasPart', 'ex:isSome', 'ex:describes', 'ex:isInput', 'ex:hasModifier', 'ex:is_a', 'ex:hasExperimentSetup', 'ex:hasSample', 'ex:performedWith', 'ex:hasConcentrations']
                           })

# now printing the dataframes ( writing the object name in the cell will print the object, like this:)
Class_df

In [ ]:
Relation_df

### Shape

as we use SHACL to validate whether a certain shape is fullfilled, we first generate a SHACL shape. for this a table in the form of a subject-predicate-object table is provided, which will then automatically be converted into a certain shape. 

To further understand the SHACL language, please refer to the [SHACL documentation](https://www.w3.org/TR/shacl/) and to our [SHACL Tutorial]().

In [ ]:
shape_validation_df = pd.DataFrame({'subject_label':['Process', 'Process', 'Process', 'ProcessStep', 'Input', 'Input', 'Input'],
                                    'predicate_label':['isSome', 'hasPart', 'isSome', 'describes', 'isInput', 'hasModifier', 'is_a'],
                                    'object_label':['ChemicalInvestigation', 'ProcessStep', 'realizedOccurent', 'Setup', 'ProcessStep', 'InputSettings', 'Specimen']
                                   })

shape_validation_df

:TODO: improve the modelling (some of the relations, I do not understand )

### Bridge

next the main feature, the bridging is designed via a table. Here four columns are used, where the "From" column represents the origin class of an instance, the "To" column represents the mapped class of an instance, the "Relation" column represents a new relations which shall be outgoing from the newly mapped instance and the "Target" column defines then the new object in a subject("To")-predicate("Relation")-object("Target") relationship which is generated for this newly bridged relation. The empty entries '-' in the "From" and "To" column are automatically interpreted as repeating the mapping but performing the creation of an additional relationship. Empty entires '-' in the "Target" and "Relation" column however just represent that no new relationships shall be generated for these instances.

In [ ]:
bridging_df = pd.DataFrame({'From':['Process', '-', 'ProcessStep', 'Input', 'InputSettings'],
                            'To':['Experiment', '-', 'ExperimentSetup', 'Sample', 'Parameters'],
                            'Relation':['hasExperimentSetup', 'hasSample', 'performedWith', 'hasConcentrations', '-'],
                            'Target':['ExperimentSetup', 'Sample', 'Parameters', 'Parameters', '-']
                           })

bridging_df

### Visualization
Next we first import all required python functions for generating the visualization and then we execute these functions and generate the visualization. 

In [ ]:
import sys
import code_for_shacl_bridges_2
from code_for_shacl_bridges_2 import generate_mermaid_chart_from_dfs

In [ ]:
mermaid_chart, markdown_version_of_mermaid_chart = generate_mermaid_chart_from_dfs(Class_df, Relation_df, shape_validation_df, bridging_df)

display(Markdown(markdown_version_of_mermaid_chart))

In [ ]:
# and now the chart (I made the mistake of installing mermaid package :( - don't do that, just install a mermaid previewer plugin for vscode, if you use it )

#display(Markdown(mermaid_chart))

mermaid_chart

As this Diagramm is quite large and initially overwhelming, a short introductions shall be given.
This diagramm consist of three main parts:
1. the Shape validation frame describes the complete SHACL shape used in validating whether a certain set of instances fullfill the criteria to be bridged into another ontology design pattern.
2. the CoreShapeInformation frame represents just the classes of the respective instances which can be bridged. So only because you have a certain set of criteria must not mean that you want to bridge all instances involved in these criterias. Therefore only a subset of instances can be bridge. which in this case are just the instances inside this CoreShapeInformation.
3. the TransformedGraph frame describes the new design pattern with its terminology, as well as its new "semantic hierarchy".
4. as the bridging can largely change the topology and hierarchy the changes are also visualised via the dotted "SHACL_bridge" relation arrows, which just serve as a visual indicator of the origin of an instance.

!! check whether for larger project graph transformers are the more performant solution, s., e.g.: 

https://graphscope.io/docs/latest/overview/intro

Here a little bit theoretical background on the graph transformers:

https://user.informatik.uni-bremen.de/hof/papers/Grace98.pdf




### Generaing he SHAFE Bridge

as we now have validated whether we have provided the shape as intended we can now generate the SHAFE Bridge artefact in form of a SHACL Shape with a SPARQL query infused in it, which uses the Construct Keyword to generate the new relations.

In [19]:
from code_for_shacl_bridges_2 import generate_shafe_from_dfs

In [ ]:
final_shacl_2 = generate_shafe_from_dfs(Class_df, Relation_df, shape_validation_df, bridging_df)

In [ ]:
print(final_shacl_2)

The SHACL Shape uses dynamically generated variables for the bridging process. Here we must first read the "WHERE" Block in order to understand the origin, before reading the "CONSTRUCT" Block, we the transformation is performed. in here You can see that the SHAFE Bridge does not generate new Instances, but rather just assertes an additional Class and additional relations to the original Instance. Only extracting the difference between the original graph and the new graph enables us to generate a semantic Artefact or "Intermediate" which can the exported/imported into the Knowledge Graph which was modeled via the bridge-targets ontology. This is done below.

In [22]:
from code_for_shacl_bridges_2 import gen_dif_and_ext_graph

In [ ]:
path_to_data = "./test_ontology_and_data_4.ttl"
# path_to_shapes = "./test_shacl_sparql_1.ttl"
path_to_shapes = "./shacl_validation_2.ttl"

data_graph = Graph()
data_graph.parse(path_to_data, format="ttl")

shape_graph = Graph()
shape_graph.parse(path_to_shapes, format="ttl")

expanded_graph, diff_graph, conforms, report_graph_dict, report_text = gen_dif_and_ext_graph(data_graph, shape_graph)
expanded_graph.serialize(destination='./expanded_graph.ttl', format='turtle')
diff_graph.serialize(destination='./diff_graph.ttl', format='turtle')

In [ ]:
with open('./diff_graph.ttl', 'r')as f:
    diff_graph_read = f.read()

print(diff_graph_read)

Above you can see the new relations and topology generated via this SHAFE Bridge. This graph also serves as the intermediate.

In [ ]:
with open('./test_ontology_and_data_4.ttl', 'r')as f:
    test_data_graph_read = f.read()

print(test_data_graph_read)

Above you can see the full graph which was generated via this tool. While the simplicity of the inputs dont allow for more advanced SPARQL queries and thereby generated relations, this tool can of course be used to also modify just the original graph. This however is not the intention of a SHACL Shape describing itself as SHAFE Bridge. As such the use of cleare terminology and documentation is advised, such that foreing 